# Entscheidungsbäume & Einführung in scikit-learn

In diesem Notebook werden wir:

1. **Einen neuen Datensatz erkunden** — den *Red Wine Quality*-Datensatz der UCI.
2. **Die Bibliothek scikit-learn kennenlernen** — insbesondere das einheitliche API-Muster `fit` / `predict` / `score`.
3. **Entscheidungsbäume für Klassifikation** trainieren, visualisieren und interpretieren.
4. **Entscheidungsbäume für Regression** anwenden und mit linearer Regression vergleichen.
5. **Überanpassung (Overfitting) und Pruning** untersuchen sowie **Kreuzvalidierung** mit scikit-learn durchführen.
6. **Verschiedene Modelle vergleichen** — Entscheidungsbaum, logistische Regression und Naive Bayes.

In den bisherigen Übungen haben wir Algorithmen und Metriken komplett von Hand mit [NumPy](https://www.numpy.org) implementiert. Diese Mal werden wir die Bibliothek [scikit-learn](https://scikit-learn.org/stable/api/) verwenden.

## Benötigte Module

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, mean_squared_error, mean_absolute_error, r2_score
)

## Reproduzierbarkeit

In [ ]:
RANDOM_SEED = 19751979

---
## Teil 1 — Der Weinqualitätsdatensatz

Der Red Wine Quality-Datensatz stammt aus dem [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/186/wine+quality) und enthält **1.599 Rotweinproben** aus dem portugiesischen Weinanbaugebiet *Vinho Verde*. Jede Probe wurde physikochemisch analysiert (11 Merkmale) und von Experten mit einer **Qualitätsnote von 0 bis 10** bewertet.

| Spaltenname | Merkmal | Beschreibung |
|---|---|---|
| `fixed acidity` | Feste Säure | Weinsäure (g/dm<sup>3</sup>) |
| `volatile acidity` | Flüchtige Säure | Essigsäure (g/dm<sup>3</sup>) — zu viel führt zu unangenehmem Geschmack |
| `citric acid` | Zitronensäure | Kann Weinen Frische verleihen (g/dm<sup>3</sup>) |
| `residual sugar` | Restzucker | Verbleibender Zucker nach der Gärung (g/dm<sup>3</sup>) |
| `chlorides` | Chloride | Salzmenge im Wein (g/dm<sup>3</sup>) |
| `free sulfur dioxide` | Freies SO<sub>2</sub> | Freies Schwefeldioxid (mg/dm<sup>3</sup>) — schützt vor Oxidation |
| `total sulfur dioxide` | Gesamtes SO<sub>2</sub> | Gesamtmenge an Schwefeldioxid (mg/dm<sup>3</sup>) |
| `density` | Dichte | Dichte des Weins (g/cm<sup>3</sup>) |
| `pH` | pH-Wert | Säuregrad (Skala 0–14) |
| `sulphates` | Sulfate | Kaliumsulfat (g/dm<sup>3</sup>) — konservierender Zusatzstoff |
| `alcohol` | Alkoholgehalt | Alkoholgehalt in Vol.-% |
| `quality` | Qualität | Bewertung durch Experten (0–10) |

<span style="font-size:x-small">Die Ersteller des Datensatzes sind P. Cortez, A. Cerdeira, F. Almeida, Telmo Matos und J. Reis. Er wird erstmals im folgenden Artikel beschrieben:<br />
Cortez et al. <a href="https://www.sciencedirect.com/science/article/abs/pii/S0167923609001377">Modeling wine preferences by data mining from physicochemical properties</a>. Decision Support Systems, 2009
</span>

In [ ]:
def parse_dataset(lines: list[str]) -> tuple[np.ndarray, list[str]]:
    """
    Parsiert den Datensatz zur Rotweinqualität aus Text.
    :param lines: Inhalt der Textdatei als `list` von `str`-Instanzen, eine pro Zeile.
    :return: Tupel der Form `(x, y, feature_names)`.
    """
    delimiter = ';'
    column_names = {
        'fixed acidity': 'Feste Säure',
        'volatile acidity': 'Flüchtige Säure',
        'citric acid': 'Zitronensäure',
        'residual sugar': 'Restzucker',
        'chlorides': 'Chloride',
        'free sulfur dioxide': 'Freies SO₂',
        'total sulfur dioxide': 'Gesamtes SO₂',
        'density': 'Dichte',
        'pH': 'pH-Wert',
        'sulphates': 'Sulfate',
        'alcohol': 'Alkoholgehalt',
        'quality': 'Qualität'}
    expected_header = [f'"{c_name}"' for c_name in column_names.keys()]
    if lines[0].strip('\r\n') != delimiter.join(expected_header):
        raise ValueError('Unerwartete Kopfzeile')
    observation_count, feature_count = len(lines) - 1, len(column_names) - 1
    x = np.empty((observation_count, feature_count), dtype=np.float64)
    y = np.empty(observation_count, dtype=np.uint16)
    for i, line in enumerate(lines[1:]):
        for j, value in enumerate(line.strip('\r\n').split(';')):
            if j < feature_count:
                x[i, j] = float(value)
            else:
                y[i] = int(value)
    return x, y, list(column_names.values())[:-1]


def extract_lines_from_url() -> list[str]:
    """
    Lädt den Datensatz zur Rotweinqualität vom UCI-Repository herunter.
    :return: Inhalt des Datensatzes als `list` von `str`-Instanzen, eine pro Zeile.
    """
    import urllib
    dataset_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
    response = urllib.request.urlopen(dataset_url)
    return response.read().decode('utf-8').splitlines()


def extract_lines_from_file() -> list[str]:
    """
    Lädt den Datensatz zur Rotweinqualität von lokaler Datei (.csv.gz).
    :return: Inhalt des Datensatzes als `list` von `str`-Instanzen, eine pro Zeile.
    """
    import gzip
    dataset_file = 'winequality-red.csv.gz'
    with gzip.open(dataset_file, 'rt', encoding='utf-8') as file_handle:
        result = file_handle.readlines()
    return result


x, y, feature_names = parse_dataset(extract_lines_from_url())
print(f'Dimensionen der Markmalsmatrix: {x.shape}')
print(f'Dimensionen des Ausgabevektors: {y.shape}')
print(f'                   Mermalnamen: {",".join(feature_names)}')
print(f'            Erster Ausgabewert: {y[0]}')
print(f'                   Erste Zeile: {x[0, :]}')

### Verteilung der Zielvariable

Schauen wir uns an, wie die Qualitätsbewertungen verteilt sind.

In [ ]:
def plot_outcome_distribution(quality_counts: np.ndarray) -> plt.Figure:
    mid_height = quality_counts.max().item() // 2
    figure, ax = plt.subplots(figsize=(5, 4), dpi=100)
    xs = np.arange(quality_counts.size)
    bars = ax.bar(xs, quality_counts, color='#000080', edgecolor='white', width=0.8)
    for bar in bars:
        height = int(bar.get_height())
        if height > mid_height:
            text_color = '#F0F0F0'
            text_y = height - 20
            text_va = 'top'
        else:
            text_color = '#080808'
            text_y = height + 10
            text_va = 'bottom'
        ax.text(bar.get_x() + bar.get_width() / 2, text_y,
                str(height), color=text_color, ha='center', va=text_va)
    ax.set(axisbelow=True, xlim=[xs[0] - 0.5, xs[-1] + 0.5], xticks=xs)
    ax.set(xlabel='Qualitätsbewertung', ylabel='Anzahl der Proben')
    ax.grid(axis='y', color='#A0A0A0', linestyle='--', linewidth=0.5)
    return figure


figure = plot_outcome_distribution(np.bincount(y))
plt.tight_layout()
plt.show(figure)

### Verteilung der Merkmale

Histogramme aller 11 physikochemischen Merkmale:

In [ ]:
def plot_distributions(x: np.ndarray, column_names: list[str]) -> plt.Figure:
    figure, axes = plt.subplots(3, 4, figsize=(15, 8), dpi=100)
    axes = axes.flatten()
    for i, (column_name, ax) in enumerate(zip(column_names, axes)):
        ax.hist(x[:, i], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
        ax.set_title(column_name, fontsize=10)
        ax.set_ylabel('Häufigkeit')
    axes[-1].set_visible(False)  # letztes (leeres) Subplot ausblenden

figure = plot_distributions(x, feature_names)
plt.tight_layout()
plt.show(figure)

### Übung: Boxplots nach Qualitätsstufe

Erstellen Sie **Boxplots des Alkoholgehalts**, gruppiert nach Qualitätsbewertung.

*Tipp:* Erstellen Sie eine Liste von Arrays, in der Jedes Element alle Alcoholgehaltswerte für Proben einer bestimmter Bewertung beinhaltet.
Das Diagramm können Sie dann mit [`ax.boxplot()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.boxplot.html) erstellen.

Gibt es einen sichtbaren Zusammenhang zwischen Alkoholgehalt und Qualität?

In [ ]:
def plot_quality_vs_alcohol(qualities: np.ndarray, alcohol: np.ndarray) -> plt.Figure:
    """
    Generiert Boxplots von Alcoholgehaltswerte, ein pro Qualitätsscore.
    
    :param qualities: Qualitätsbewertungen von allen Proben im Datensatz als Array der
                      Form `(n,)`.
    :param alcohol: Alcoholgehalt in % von allen Proben im Datensatz als Array der Form
                    `(n,)`.
    :return: das neu erstellte Figure-Objekt.
    """
    figure, ax = plt.subplots(figsize=(5, 3.5), dpi=100)

    q_unique, a_by_quality = [], []
    for quality in range(qualities.max() + 1):
        ii = np.where(qualities == quality)[0]
        if ii.size != 0:
            q_unique.append(quality)
            a_by_quality.append(alcohol[ii])

    outlier_style = {'marker': 'x', 'markersize': 4, 'markeredgecolor': '#909090'}
    _ = ax.boxplot(a_by_quality, tick_labels=q_unique, flierprops=outlier_style)
    ax.set(axisbelow=True)
    ax.set(xlabel='Qualitätsbewertung', ylabel='Alkoholgehalt (Vol.-%)')
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    return figure


figure = plot_quality_vs_alcohol(y, x[:, feature_names.index('Alkoholgehalt')])
plt.tight_layout()
plt.show(figure)

### Korrelationsmatrix

Zum Abschluss der Erkundung schauen wir uns die paarweisen Korrelationen an:

In [ ]:
def plot_correlations(correlations: np.ndarray, variable_names: list[str]) -> plt.Figure:
    """
    Erstellt ein Diagramm aller paarweisen Korrelationen zwischen einer Reihe von Variablen.
    :param correlations: Korrelationskoeffiziente als symmetrischer Array der Form `(n,n)`.
    :param variable_names: Namen der Variablen als Liste von Länge `n`.
    :return: das neu erstellte Figure-Objekt.
    """
    figure, ax = plt.subplots(figsize=(9, 8), dpi=100)
    im = ax.imshow(correlations, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

    # Korrelationswerte in die Zellen schreiben
    n = len(variable_names)
    for i in range(n):
        for j in range(n):
            value = correlations[i, j]
            text_color = 'white' if abs(value) > 0.55 else 'black'
            ax.text(j, i, f'{value:.2f}', ha='center', va='center',
                    fontsize=7.5, color=text_color)

    figure.colorbar(im, ax=ax, shrink=0.8, label='Korrelationskoeffizient')
    ax.set(xticks=range(n), yticks=range(n))
    ax.set_xticklabels(variable_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(variable_names, fontsize=9)
    return figure


correlations = np.corrcoef(np.vstack((x.T, y.reshape(1, -1))))
variable_names = feature_names + ['Qualität']
figure = plot_correlations(correlations, variable_names)
plt.tight_layout()
plt.show(figure)
del correlations, variable_names, figure

---
## Teil 2: Einführung in scikit-learn

### Das Estimator-API

**scikit-learn** (kurz: `sklearn`) bietet eine einheitliche Schnittstelle für alle Modelle.
Das Grundmuster sieht immer gleich aus:

```python
from sklearn.some_module import SomeModel

# 1) Modell erstellen (mit Hyperparametern)
model = SomeModel(hyperparameter1=wert1, hyperparameter2=wert2)

# 2) Modell an Trainingsdaten anpassen
model.fit(x_train, y_train)

# 3) Vorhersagen treffen
y_hat = model.predict(x_test)

# 4) Modell bewerten
score = model.score(x_test, y_test)
```

> **Wichtig:** Egal ob lineare Regression, Entscheidungsbaum oder ein anderes Modell — die Methoden `fit()`, `predict()` und `score()` bleiben immer gleich. Nur der Modellname ändert sich!

### Daten vorbereiten: Feature-Matrix und Zielvektor

In scikit-learn erwarten alle Modelle:
- **`x`** — eine Feature-Matrix der Form `(n_samples, n_features)`, also ein 2D-Array
- **`y`** — einen Zielvektor der Form `(n_samples,)`, also ein 1D-Array

### Train-Test-Split

Bisher haben wir den Train-Test-Split manuell implementiert. In scikit-learn heißt die entsprechende Funktion [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html):

In [ ]:
x_train, x_test, y_train, y_test =\
    train_test_split(x, y, test_size=0.2, random_state=RANDOM_SEED)

print(f'Trainingsdaten: {x_train.shape[0]} Proben')
print(f'     Testdaten: {x_test.shape[0]} Proben')

### Lineare Regression mit scikit-learn

Das untenstehende Code-Snippet trainiert ein lineares Regressionsmodell zur Vorhersage von Qualitätswerten und evaluiert es anschließend auf dem Testset.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(x_train, y_train)

print('Evaluation der linearen Regression auf Testdaten')
y_hat_lr = lr_model.predict(x_test)
print(f'R²-Score: {lr_model.score(x_test, y_test):.4f}')
print(f'     MAE: {mean_absolute_error(y_test, y_hat_lr):.4f}')
print(f'    RMSE: {np.sqrt(mean_squared_error(y_test, y_hat_lr)):.4f}')

### Übung: Logistische Regression mit `scikit-learn`

Erstellen Sie eine binäre Klassifikationsaufgabe: Wein ist *gut* (`1`) wenn `Qualität >= 7`, sonst *nicht gut* (`0`).

Führen Sie folgende Schritte durch:
1. Erstellen Sie den binären Zielvektor `y_binary`.
2. Teilen Sie die Daten in Trainings- und Testdaten auf. Der Testsatz sollte 20% der Daten beinhalten.
3. Erstellen Sie ein [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)-Modell mit `max_iter=1000` und setzen Sie auch einen Wert für `random_state`.
4. Trainieren Sie das Modell und berechnen Sie seine Genauigkeit (Accuracy) auf den Testdaten mit `model.score()`.

*Tipp:* Die Schritte sind analog zur linearen Regression oben — nur der Modellname ändert sich.

In [ ]:
# 1) Binärer Zielvektor
y_binary = (y >= 7).astype(np.uint16)
print(f'Klasse 0: {np.sum(y_binary == 0)} Proben')
print(f'Klasse 1: {np.sum(y_binary == 1)} Proben')

# 2) Train-Test split
x_train_b, x_test_b, y_train_b, y_test_b =\
    train_test_split(x, y_binary, test_size=0.2, random_state=RANDOM_SEED)

# 3) Logistische Regression erstellen
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)

# 4) Trainieren und Genauigkeit berechnen
log_reg.fit(x_train_b, y_train_b)
accuracy_train = log_reg.score(x_train_b, y_train_b)
accuracy_test = log_reg.score(x_test_b, y_test_b)
print(f'Genauigkeit auf Trainingsdaten: {accuracy_train:.4f}')
print(f'     Genauigkeit auf Testdaten: {accuracy_test:.4f}')

> Die hohe Genauigkeit ist hier trügerisch — da nur ~13.5% der Weine als *gut* eingestuft werden, würde ein Modell, das immer *nicht gut* vorhersagt, schon ~86.5% Genauigkeit erreichen.

---
## Teil 3: Entscheidungsbaum

### Klassifikationsziel definieren

Statt der binären Einteilung erstellen wir nun **drei Qualitätsklassen**:

| Qualitätsbewertung | Klasse |
|---|---|
| 3, 4, 5 | Niedrig |
| 6 | Mittel |
| 7, 8 | Hoch |

So haben wir ein Mehrklassen-Problem mit einer etwas ausgewogeneren Verteilung.

In [ ]:
def quality_to_class(qualities: np.ndarray) -> np.ndarray:
    """Qualitätsbewertung in Klasse umwandeln.
    :param qualities: Qualitätsbewertungen als Array der Form `(n,)` mit Zahlen zwischen 0
                      und 10.
    :return: Tupel der Form `(classes, class_names)`, wobei
             `classes` ein Array der Form `(n,)` mit Klassenindizes ist, und
             `class_names` eine Liste mit den 3 Klassennamen ist.
    """
    result = np.ones(qualities.size, dtype=np.uint16)
    result[qualities <= 5] = 0
    result[7 <= qualities] = 2
    return result, ['Niedrig', 'Mittel', 'Hoch']


y_class, class_names = quality_to_class(y)

# Verteilung der Klassen
print(f'Klassenverteilung: {np.bincount(y_class)}')

### Daten aufteilen und ersten Entscheidungsbaum trainieren

In [ ]:
# Train-Test-Split für die Klassifikationsaufgabe
x_train_c, x_test_c, y_train_c, y_test_c =\
    train_test_split(x, y_class, test_size=0.2, random_state=RANDOM_SEED, stratify=y_class)

print(f'Trainingsdaten: {x_train.shape[0]} Proben')
print(f'     Testdaten: {x_test.shape[0]} Proben')

In [ ]:
# Entscheidungsbaum ohne Beschränkungen (volle Tiefe)
tree_clf_full = DecisionTreeClassifier(random_state=RANDOM_SEED)
tree_clf_full.fit(x_train_c, y_train_c)

train_acc = tree_clf_full.score(x_train_c, y_train_c)
test_acc = tree_clf_full.score(x_test_c, y_test_c)

print(f'Genauigkeit auf Trainingsdaten: {train_acc:.1%}')
print(f'     Genauigkeit auf Testdaten: {test_acc:.1%}')
print(f'               Tiefe des Baums: {tree_clf_full.get_depth()}')
print(f'                Anzahl Blätter: {tree_clf_full.get_n_leaves()}')

### Den Baum visualisieren

Die Funktion [`plot_tree`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html) aus scikit-learn erlaubt es uns, den Entscheidungsbaum direkt zu zeichnen.

Da der vollständige Baum zu groß ist, trainieren wir zunächst einen flachen Baum mit `max_depth=3`:

In [ ]:
# Flacher Baum für Visualisierung
tree_clf_shallow = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED)
tree_clf_shallow.fit(x_train_c, y_train_c)

_, ax = plt.subplots(figsize=(16, 8), dpi=100)
plot_tree(
    tree_clf_shallow,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax,
    impurity=True,     # Gini-Wert anzeigen
    proportion=False)
plt.tight_layout()
plt.show()

test_acc_shallow = tree_clf_shallow.score(x_test_c, y_test_c)
print(f'Genauigkeit auf Testdaten (max_depth=3): {test_acc_shallow:.2%}')

### Den Baum lesen und interpretieren

Jeder Knoten zeigt:
- **Bedingung**: z. B. `Alkoholgehalt <= 10.525` — das ist die Splitregel.
- **gini**: Der Gini-Unreinheitswert (0 = rein, max ≈ 0.67 bei 3 Klassen).
- **samples**: Anzahl der Trainingsbeispiele in diesem Knoten.
- **value**: Verteilung der Klassen `[Niedrig, Mittel, Hoch]`.
- **class**: Die Mehrheitsklasse in diesem Knoten.

> **Erinnerung aus der Vorlesung:** Der Gini-Index ist definiert als:
> $$G(t) = 1 - \sum_{k=1}^{K} p_k^2$$
> wobei $p_k$ der Anteil der Klasse $k$ im Knoten $t$ ist.

### Hausaufgabe: Baum mit anderer Tiefe

1. Trainieren Sie einen Entscheidungsbaum mit `max_depth=5`.
2. Visualisieren Sie den Baum. Nutzen Sie `fontsize=8`, da der Baum größer wird.
3. Berechnen Sie die Genauigkeit auf den Testdaten.
4. Vergleichen Sie: Ist der tiefere Baum besser?

In [ ]:
# Flacher Baum für Visualisierung
tree_clf_5 = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_SEED)
tree_clf_5.fit(x_train_c, y_train_c)

_, ax = plt.subplots(figsize=(16, 8), dpi=100)
plot_tree(
    tree_clf_5,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax,
    impurity=True,     # Gini-Wert anzeigen
    proportion=False)
plt.tight_layout()
plt.show()

test_acc_5 = tree_clf_5.score(x_test_c, y_test_c)
print(f'Genauigkeit auf Testdaten (max_depth=5): {test_acc_5:.2%}')

### Merkmalswichtigkeit (Feature Importance)

Entscheidungsbäume berechnen automatisch, wie wichtig jedes Merkmal für die Klassifikation ist. Die Wichtigkeit basiert darauf, wie stark ein Merkmal die Unreinheit (Gini) über alle Splits reduziert.

Das untenstehende Code-Snippet visualisiert die Merkmalswichtigkeit des Baums mit `max_depth = 3`.

In [ ]:
def plot_importances(feature_names: list[str], values: np.ndarray) -> plt.Figure:
    """
    Stellt Merkmalswichtigkeiten als horizontale Balken dar.
    :param feature_names: Namen aller Merkmale als Liste von Länge `n`.
    :param values: Wichtigkeitswerte als Array der Form `(n,)`.
    :return: das neu erstellte Figure-Objekt.
    """
    d = len(feature_names)
    max_value = values.max()
    figure, ax = plt.subplots(figsize=(8, 4.5), dpi=100)
    bars = ax.barh(range(d), values, color='steelblue', edgecolor='white')
    for i, value in enumerate(values):
        text_color = '#F3F3F3' if value > max_value / 20 else '#505050'
        ax.text(0.01, i, f'{value:.3f}', color=text_color, ha='left', va='center')
    ax.set(axisbelow=True, xlabel='Merkmalswichtigkeit')
    ax.set(ylim=[-0.5, d - 0.5], yticks=range(d), yticklabels=feature_names)
    ax.grid(axis='x', color='#A0A0A0', linestyle='--', linewidth=0.5)
    return figure


importances = tree_clf_shallow.feature_importances_
sorted_indices = np.argsort(importances)
names_sorted = [feature_names[i] for i in sorted_indices]
figure = plot_importances(names_sorted, importances[sorted_indices])
plt.tight_layout()
plt.show(figure)
del importances, sorted_indices, names_sorted, figure

### Übung: Konfusionsmatrix und Klassifikationsbericht

1. Berechnen Sie die Vorhersagen des Entscheidungsbaums (`max_depth=5`) auf den Testdaten.
2. Erstellen Sie eine *Konfusionsmatrix* (auch Wahrheitsmatrix genannt) mit [`ConfusionMatrixDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html).
3. Geben Sie den Klassifikationsbericht ([`classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)) aus.

*Tipp:*
```python
y_hat = model.predict(x_test)
cm = confusion_matrix(y_test, y_hat)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(cmap='Blues')
```

In [ ]:
# Vorhersagen
y_hat_c = tree_clf_5.predict(x_test_c)

# Konfusionsmatrix
cm = confusion_matrix(y_test_c, y_hat_c)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
_, ax = plt.subplots(figsize=(5, 4), dpi=100)
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set(xlabel='Vorhergesagte Klasse', ylabel='Tatsächliche Klasse')
plt.tight_layout()
plt.show()

# Klassifikationsbericht
print('Klassifikationsbericht:')
print(classification_report(y_test_c, y_hat_c, target_names=class_names))

---
## Teil 4: Entscheidungsbaum für Regression

Bisher haben wir Entscheidungsbäume für Klassifikation verwendet. Nun verwenden wir denselben Datensatz, um die Qualitätsnote als kontinuierliche Zahl vorherzusagen — also eine *Regression*. Die entsprechende Klasse in sklearn ist [`DecisionTreeRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html).

### Wie funktioniert ein Regressionsbaum?

Der Unterschied zum Klassifikationsbaum:
- Statt der Gini-Unreinheit minimiert ein Regressionsbaum den *mittleren quadratischen Fehler (MSE)* in jedem Knoten.
- Die Vorhersage in jedem Blatt ist der Mittelwert der Zielwerte aller Trainingsbeispiele, die in diesem Blatt landen.
- Da die Vorhersage pro Blatt konstant ist, entsteht eine *treppenförmige* Vorhersagefunktion.

In [ ]:
# Trainings- und Testsatz erstellen
x_train_r, x_test_r, y_train_r, y_test_r =\
    train_test_split(x, y, test_size=0.2, random_state=RANDOM_SEED)

# Regressionsbaum trainieren
tree_reg = DecisionTreeRegressor(max_depth=4, random_state=RANDOM_SEED)
tree_reg.fit(x_train_r, y_train_r)

# Modell evaluieren
y_hat_r = tree_reg.predict(x_test_r)
print(f'R²-Score: {tree_reg.score(x_test_r, y_test_r):.4f}')
print(f'     MAE: {mean_absolute_error(y_test_r, y_hat_r):.4f}')
print(f'    RMSE: {np.sqrt(mean_squared_error(y_test_r, y_hat_r)):.4f}')

### Vorhersagen vs. tatsächliche Werte

In [ ]:
def plot_true_predicted(y_true: np.ndarray, y_predicted: np.ndarray) -> plt.Figure:
    """
    Erstellt ein Streudiagramm der wahren und vorhergesagten Werte.
    :param y_true: Wahre Qualitätswerte als Array der Form `(n,)`.
    :param y_predicted: Vorhergesagte Werte als Array der Form `(n,)`.
    :return: das neu erstellte Figure-Objekt.
    """
    figure, ax = plt.subplots(figsize=(4.5, 4.5), dpi=100)
    ax.scatter(y_true, y_predicted, alpha=0.75, color='steelblue', edgecolors='white', s=40)
    value_min = min(y_true.min(), y_predicted.min()) - 0.5
    value_max = max(y_true.max(), y_predicted.max()) + 0.5
    value_range = [value_min, value_max]
    ax.plot(value_range, value_range, '--', c='#A0A0A0')  # idealle Vorhersage
    ax.set(aspect='equal', axisbelow=True, xlim=value_range, ylim=value_range)
    ax.set(xlabel='Tatsächliche Qualität', ylabel='Vorhergesagte Qualität')
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    return figure


figure = plot_true_predicted(y_test_r, y_hat_r)
plt.tight_layout()
plt.show(figure)

### Der Treppeneffekt — Visualisierung in 1D

Um zu verstehen, wie ein Regressionsbaum vorhersagt, schauen wir uns die Vorhersage entlang eines einzelnen Merkmals an. Wir verwenden den Alkoholgehalt, da er die stärkste Korrelation mit der Qualität hat.

In [ ]:
def plot_trees_1d(x: np.ndarray, y: np.ndarray, feature_names: list[str], feature_name: str) -> plt.Figure:
    """
    Plottet die Vorhersagen von einpaar eindimensionalen Regressionsbäumen mit
    unterschiedlichen Tiefen.
    :param x: Kompletten Datensatz als Array der Form `(n,d)`.
    :param y: Ausgabewerte als Array der Form `(n,)`.
    :param feature_names: Merkmalsnamen als `list` der Länge `n`.
    :param feature_name: Name der zu verwendenden Merkmal. Alle anderen Merkmale werden
                         ignoriert.
    :return: das neu erstellte Figure-Objekt.
    """
    # Datensätze mit nur einem Feature erstellen
    x = x[:, [feature_names.index(feature_name)]]
    x_train_a, x_test_a, y_train_a, y_test_a =\
        train_test_split(x, y, test_size=0.2, random_state=RANDOM_SEED)

    depths = [1, 3, 8]
    x_grid = np.linspace(x.min(), x.max(), 500).reshape(-1, 1)
    figure, axes = plt.subplots(1, len(depths), figsize=(18, 5), dpi=100, sharey='all')

    for ax, depth in zip(axes, depths):
        ax.scatter(x_train_a, y_train_a, alpha=0.3, s=15, color='steelblue',
                   label='Trainingsdaten')
        tree_1d = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_SEED)
        tree_1d.fit(x_train_a, y_train_a)
        y_grid = tree_1d.predict(x_grid)
        ax.plot(x_grid, y_grid, color='#D9534F', linewidth=2.5,
                label=f'Baum (Tiefe {depth})')
        ax.set(title=f'max_depth = {depth}', xlabel=feature_name)
        if depth == depths[0]:
            ax.set_ylabel('Qualität')
        ax.legend(fontsize=9)
    return figure


figure = plot_trees_1d(x, y, feature_names, 'Alkoholgehalt')
plt.tight_layout()
plt.show(figure)

### Vergleich mit linearer Regression

Vergleichen wir den Regressionsbaum mit der linearen Regression auf derselben 1D-Aufgabe:

In [ ]:
def plot_lr_tree_1d(x: np.ndarray, y: np.ndarray, feature_names: list[str], feature_name: str) -> plt.Figure:
    """
    Plottet die Vorhersagen von einem linearen Regressionsmodell und einem eindimensionalen
    Regressionssbaum.
    :param x: Kompletten Datensatz als Array der Form `(n,d)`.
    :param y: Ausgabewerte als Array der Form `(n,)`.
    :param feature_names: Merkmalsnamen als `list` der Länge `n`.
    :param feature_name: Name der zu verwendenden Merkmal. Alle anderen Merkmale werden
                         ignoriert.
    :return: das neu erstellte Figure-Objekt.
    """
    # Datensätze mit nur einem Feature erstellen
    x = x[:, [feature_names.index(feature_name)]]
    x_train_a, x_test_a, y_train_a, y_test_a =\
        train_test_split(x, y, test_size=0.2, random_state=RANDOM_SEED)
    
    # Lineare Regression und Regressionsbaum trainieren
    lr_1d = LinearRegression()
    lr_1d.fit(x_train_a, y_train_a)
    lr_1d_score = lr_1d.score(x_test_a, y_test_a)
    tree_1d = DecisionTreeRegressor(max_depth=4, random_state=RANDOM_SEED)
    tree_1d.fit(x_train_a, y_train_a)
    tree_1d_score = tree_1d.score(x_test_a, y_test_a)

    x_grid = np.linspace(x.min(), x.max(), 500).reshape(-1, 1)
    figure, axes = plt.subplots(1, 2, figsize=(9, 4), dpi=100, sharey='all')

    axes[0].scatter(x_train_a, y_train_a, alpha=0.3, s=15, color='steelblue')
    axes[0].plot(x_grid, lr_1d.predict(x_grid), color='#D9534F', linewidth=2)
    axes[0].set_title(f'Lineare Regression (R² = {lr_1d_score:.3f})')
    axes[0].set(xlabel=feature_name, ylabel='Qualität')

    axes[1].scatter(x_train_a, y_train_a, alpha=0.3, s=15, color='steelblue')
    axes[1].plot(x_grid, tree_1d.predict(x_grid), color='#D9534F', linewidth=2)
    axes[1].set_title(f'Regressionsbaum (R² = {tree_1d_score:.3f})')
    axes[1].set(xlabel=feature_name)
    return figure


figure = plot_lr_tree_1d(x, y, feature_names, 'Alkoholgehalt')
plt.tight_layout()
plt.show(figure)

### Übung: Regressionsbaum optimieren

1. Trainieren Sie `DecisionTreeRegressor`-Modelle mit `max_depth` = 2, 3, 4, 5, 6, 8, 10 und `None` (unbeschränkt) auf den Trainingssatz mit allen Merkmalen.
2. Berechnen Sie für jedes Modell den *R<sup>2</sup>-Score* auf den *Trainings-* und *Testdaten*.
3. Erstellen Sie ein Liniendiagramm mit `max_depth` auf der x-Achse und R<sup>2</sup> auf der y-Achse, jeweils eine Linie für Training und Test.

*Tipps:* Nutzen Sie `x_train_r, x_test_r, y_train_r, y_test_r`, die wir schon erstellt haben. Wenn Sie die Funktion `plot_scores` aufrufen, setzen Sie `hyperparameter_name='Maximale Tiefe', score_name='$R^2$'`. 

In [ ]:
def plot_scores(scores: np.ndarray, model_names: list[str],
                hyperparameter_name: str, score_name: str) -> plt.Figure:
    """
    Erstellt Kurven der Trainings- und Testscores.
    
    :param scores: Werte zum plotten als Array der Form `(n, 2)`; die erste Spalte
                   sollte Trainings-, und die zweite - Testscores beinhalten.
    :param model_names: Modelnamen als Liste der Länge `n`.
    :param hyperparameter_name: Name des Hyperparameters (Label der x-Achse).
    :param score_name: Maßname.
    :return: das neu erstellte Figure-Objekt.
    """
    figure, ax = plt.subplots(figsize=(6, 3), dpi=100)
    xs = np.arange(scores.shape[0])
    ax.plot(xs, scores[:, 0], 'o-', color='steelblue', linewidth=2, label='Training')
    ax.plot(xs, scores[:, 1], 'o--', color='#D9534F', linewidth=2, label='Test')
    ax.set(axisbelow=True, xlabel=hyperparameter_name, ylabel=score_name) 
    ax.set(xlim=[xs[0] - 0.2, xs[-1] + 0.2], xticks=xs, xticklabels=model_names)
    ax.legend()
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    return figure


depths = [2, 3, 4, 5, 6, 8, 10, None]
depth_names = [' ' if d is None else str(d) for d in depths]
scores = np.empty((len(depths), 2), dtype=np.float64)
for i, depth in enumerate(depths):
    tree = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_SEED)
    tree.fit(x_train_r, y_train_r)
    scores[i, 0] = tree.score(x_train_r, y_train_r)
    scores[i, 1] = tree.score(x_test_r, y_test_r)
figure = plot_scores(scores, depth_names, hyperparameter_name='Maximale Tiefe', score_name='$R^2$')
plt.tight_layout()
plt.show(figure)

---
## Teil 5: Überanpassung, Pruning und Kreuzvalidierung

### Überanpassung bei Entscheidungsbäumen

Entscheidungsbäume neigen besonders stark zur Überanpassung (Overfitting):
- Ein unbeschränkter Baum kann die Trainingsdaten *perfekt* lernen (jedes Blatt enthält genau ein Beispiel).
- Auf neuen Daten generalisiert er aber schlecht.

Dies entspricht dem *Bias-Varianz-Tradeoff* aus der Vorlesung:
- **Niedriges `max_depth`** -> hoher Bias, niedrige Varianz (Underfitting)
- **Hohes `max_depth`** -> niedriger Bias, hohe Varianz (Overfitting)

### Pruning

In scikit-learn können wir den Baum über verschiedene Hyperparameter beschneiden:

| Hyperparameter | Beschreibung |
|---|---|
| `max_depth` | Maximale Tiefe des Baums |
| `min_samples_split` | Minimale Anzahl an Proben, um einen Knoten zu teilen |
| `min_samples_leaf` | Minimale Anzahl an Proben in einem Blatt |
| `max_leaf_nodes` | Maximale Anzahl an Blättern |

### Übung: Trainings- vs. Testgenauigkeit bei verschiedenen Tiefen

Wiederholen Sie die Analyse aus der vorherigen Übung für die Klassifikationsaufgabe. Trainieren Sie Entscheidungsbäume mit maximale Tiefe $1, 2, \ldots, 19, 20$.

In [ ]:
depths = list(range(1, 21))
depth_names = [' ' if d is None else str(d) for d in depths]
scores = np.empty((len(depths), 2), dtype=np.float64)
for i, depth in enumerate(depths):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_SEED)
    tree.fit(x_train_c, y_train_c)
    scores[i, 0] = tree.score(x_train_c, y_train_c)
    scores[i, 1] = tree.score(x_test_c, y_test_c)
figure = plot_scores(scores, depth_names, hyperparameter_name='Maximale Tiefe', score_name='Genauigkeit')
plt.tight_layout()
plt.show(figure)

### Übung: Einfluss von `min_samples_leaf`

1. Trainieren Sie Entscheidungsbäume ohne `max_depth`-Beschränkung, aber mit `min_samples_leaf` = 1, 2, 5, 10, 20, 50, 100.
2. Berechnen Sie Trainings- und Testgenauigkeit.
3. Erstellen Sie ein Liniendiagramm.

Welcher Wert für `min_samples_leaf` liefert die beste Testgenauigkeit?

In [ ]:
leaf_sizes = [1, 2, 5, 10, 20, 50, 100]
size_names = [str(l) for l in leaf_sizes]

scores = np.empty((len(leaf_sizes), 2), dtype=np.float64)
for i, leaf_size in enumerate(leaf_sizes):
    tree = DecisionTreeClassifier(min_samples_leaf=leaf_size, random_state=RANDOM_SEED)
    tree.fit(x_train_c, y_train_c)
    scores[i, 0] = tree.score(x_train_c, y_train_c)
    scores[i, 1] = tree.score(x_test_c, y_test_c)
    
figure = plot_scores(scores, size_names, hyperparameter_name='Maximale Blattgröße', score_name='Genauigkeit')
plt.tight_layout()
plt.show(figure)

best_size = leaf_sizes[np.argmax(scores[:, 1])]
print(f'Beste Testgenauigkeit bei min_samples_leaf = {best_size}')
print(f'                               Genauigkeit = {scores[:, 1].max():.1%}')

### Kreuzvalidierung mit scikit-learn

scikit-learn bietet die Funktion `cross_val_score` für Kreuzvalidierung:

```python
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
```

Dieses Verfahren:
1. Teilt die Daten in `cv` gleich große Teile (Folds).
2. Trainiert das Modell `cv`-mal, jedes Mal mit einem anderen Fold als Testdaten.
3. Gibt die Scores aller Folds zurück.

In [ ]:
# 5-Fold Kreuzvalidierung für den Baum mit max_depth=5
fold_count = 5
tree_cv = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_SEED)
cv_scores = cross_val_score(tree_cv, x, y_class, cv=fold_count, scoring='accuracy')

print(f'Kreuzvalidierungsergebnisse ({fold_count} Folds):')
for i, score in enumerate(cv_scores):
    print(f'    Fold {i + 1}: {score:.4f}')
print(f'Mittelwert: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

### Übung: Optimale Baumtiefe per Kreuzvalidierung

Verwenden Sie 5-Fold Kreuzvalidierung, um die optimale `max_depth` zu finden:

1. Testen Sie `max_depth` von 1 bis 20.
2. Berechnen Sie für jede Tiefe den *mittleren CV-Score* und die *Standardabweichung*.
3. Erstellen Sie ein Diagramm mit dem mittleren CV-Score ± eine Standardabweichung (als [schattierter Bereich](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.fill_between.html)).
4. Welche Tiefe würden Sie wählen?

*Tipp für den schattierten Bereich:*
```python
ax.fill_between(x, mean - std, mean + std, alpha=0.2)
```

In [ ]:
fold_count = 5
depths = list(range(1, 21))
depth_names = [' ' if d is None else str(d) for d in depths]
scores = np.empty((len(depths), 2), dtype=np.float64)  # (depth, [mean Acc, std Acc])
for i, depth in enumerate(depths):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_SEED)
    cv_scores = cross_val_score(tree, x, y_class, cv=fold_count, scoring='accuracy')
    scores[i, 0] = cv_scores.mean()
    scores[i, 1] = cv_scores.std()

index_best_depth = np.argmax(scores[:, 0])

figure, ax = plt.subplots(figsize=(8, 4), dpi=100)
xs = np.arange(scores.shape[0])
ax.plot(xs, scores[:, 0], '-', color='steelblue', linewidth=2)
ax.fill_between(xs, scores[:, 0] - scores[:, 1], scores[:, 0] + scores[:, 1],
                color='steelblue', alpha=0.2)
ax.axvline(index_best_depth, alpha=0.7, color='gray', linestyle=':')
ax.set(xlim=[xs[0], xs[-1]], xticks=xs, xticklabels=depth_names)
ax.set(axisbelow=True, xlabel='Maximale Tiefe', ylabel='Genauigkeit (5-fold CV)')
ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show(figure)


---
## Teil 6: Modellvergleich

Früher haben wir lineare Regression, logistische Regression und Naive Bayes von Hand implementiert. Hier nutzen wir scikit-learn, um alle Modelle auf denselben Daten zu vergleichen.

Ein großer Vorteil von sklearn ist, dass das API konsenquent ist. Wir tauschen nur die Modellklassen aus.

In [ ]:
# Modelle definieren
models = {
    'Entscheidungsbaum': DecisionTreeClassifier(max_depth=4, random_state=RANDOM_SEED),
    'Logistische Regression': LogisticRegression(max_iter=4000, random_state=RANDOM_SEED),
    'Naive Bayes (Gauss)': GaussianNB()
}

# Ergebnisse sammeln
results = {}

for model_name, model in models.items():
    # Kreuzvalidierung
    cv_scores = cross_val_score(model, x, y_class, cv=5, scoring='accuracy')
    
    # Auf Train/Test trainieren für die Konfusionsmatrix
    model.fit(x_train_c, y_train_c)
    y_hat = model.predict(x_test_c)
    test_accuracy = accuracy_score(y_test_c, y_hat)
    
    results[model_name] = {
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'test_acc': test_accuracy,
        'y_hat': y_hat,
    }
    
    print(model_name)
    print(f'       CV-Score: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print(f'Testgenauigkeit: {test_acc:.4f}')
    print()

### Konfusionsmatrizen im Vergleich

In [ ]:
figure, axes = plt.subplots(1, len(results), figsize=(15, 5), dpi=100)

for i, (model_name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test_c, res['y_hat'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap='Blues', ax=axes[i], colorbar=False, values_format='d')
    axes[i].set_title(f'{model_name}\nGenauigkeit: {res['test_acc']:.3f}')
    axes[i].set(xlabel='Vorhergesagte Klasse')
    axes[i].set_ylabel('Tatsächliche Klasse' if i == 0 else '')

plt.tight_layout()
plt.show(figure)
del figure, axes, i, model_name, res, cm, disp

### CV-Scores im Vergleich

In [ ]:
figure, ax = plt.subplots(figsize=(7, 4), dpi=100)

model_names = list(results.keys())
colors = ['#5CB85C', '#F0AD4E', '#5BC0DE']
cv_means = [res['cv_mean'] for res in results.values()]
cv_devs = [res['cv_std'] for res in results.values()]
bars = ax.bar(model_names, cv_means, yerr=cv_devs, capsize=8,
              color=colors, edgecolor='white', width=0.75)
for bar, m in zip(bars, cv_means):
    ax.text(bar.get_x() + bar.get_width() / 2, 0.02,
            f'{m:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set(axisbelow=True, ylabel='Mittlere Genauigkeit (5-Fold CV)', ylim=[0, 1])
ax.grid(axis='y', color='#A0A0A0', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show(figure)
del model_names, colors, cv_means, cv_devs, bars, bar, m